In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob

In [9]:
from pathlib import Path
import re

folder = Path("/Users/jingying/Documents/IDX Exchange/py/csv")


files = []

for file in folder.glob("CRMLSSold*.csv"):
    if re.fullmatch(r"CRMLSSold\d{6}(?:_filled)?.csv", file.name):
        files.append(file)
        
files.sort(key=lambda file: file.name[9:15])


df = []

for file in files:
    monthly_data = pd.read_csv(file, low_memory=False)
    monthly_data["Date"] = file.name[9:15]
    df.append(monthly_data)
    
combined_sold_data = pd.concat(df, ignore_index=True)
combined_sold_data.shape

(615707, 85)

## Restirct analysis to Residential and SingleFamilyResidence

In [11]:
res_sfr_combined_sold_data = combined_sold_data[(combined_sold_data["PropertyType"]=="Residential") & (combined_sold_data["PropertySubType"]=="SingleFamilyResidence")]
res_sfr_combined_sold_data.shape

(309735, 85)

## 1. Removing Records That Cannot Be True

## 1.1 Logical impossibilities:

### Check target value (ClosePrice)

In [35]:
data = res_sfr_combined_sold_data.copy() 
data = data.dropna(subset=["ClosePrice"]) #remove close price = NA
data = data[data["ClosePrice"] > 0] #remove close price = 0
data.shape

(309732, 85)

In [77]:
(309735-309732)/309735 #Log the count and percentage of records removed at each cleaning step 

9.68569906533004e-06

### Check date consistency

In [37]:
# 3 Date types
data["CloseDate"] = pd.to_datetime(data["CloseDate"])
data["ListingContractDate"] = pd.to_datetime(data["ListingContractDate"])
data["PurchaseContractDate"] = pd.to_datetime(data["PurchaseContractDate"])
data[["CloseDate", "ListingContractDate", "PurchaseContractDate"]]

,CloseDate,ListingContractDate,PurchaseContractDate
5,2024-01-05,2021-03-08,2021-06-30
9,2024-01-05,2021-03-08,2021-11-18
28,2024-01-02,2023-11-15,2023-11-15
29,2024-01-22,2023-11-17,2023-12-19
31,2024-01-31,2024-01-31,2024-03-04
...,...,...,...
615673,2026-04-30,2024-05-02,2024-11-26
615674,2026-04-01,2024-04-26,2026-02-25
615675,2026-04-01,2024-04-18,2026-02-28
615677,2026-04-13,2024-04-08,2026-03-12


In [43]:
#remove rows where CloseDate is earlier than ListingContractDate
invalid_order = data["CloseDate"] < data["ListingContractDate"]
filtered_data = data[~invalid_order].copy()
filtered_data.shape

(309688, 85)

In [80]:
(309732-309688)/309732 #Log the count and percentage of records removed at each cleaning step 

0.0001420582955587411

In [49]:
#remove rows where PurchaseContractDate is earlier than ListingContractDate
invalid_order2 = filtered_data["PurchaseContractDate"] < filtered_data["ListingContractDate"]
filtered_data = filtered_data[~invalid_order2]
filtered_data.shape

(309543, 85)

In [82]:
(309688-309543)/309688 #Log the count and percentage of records removed at each cleaning step 

0.0004682131693833794

### Check square footage

In [51]:
#remove rows with zero or negative square footage
filtered_data = filtered_data[(filtered_data["LivingArea"] > 0) & (filtered_data["LotSizeSquareFeet"] > 0)]
filtered_data.shape

(303795, 85)

In [84]:
(309543-303795)/309543 #Log the count and percentage of records removed at each cleaning step 

0.018569310241226582

### Check bedroom/bathroom counts inconsistent with living area

In [65]:
filtered_data[["LivingArea", "BedroomsTotal", "BathroomsTotalInteger"]]

,LivingArea,BedroomsTotal,BathroomsTotalInteger
28,3736.0,4.0,4.0
29,2100.0,3.0,0.0
31,2442.0,4.0,3.0
37,1917.0,3.0,3.0
38,2200.0,3.0,2.0
...,...,...,...
615673,944.0,3.0,1.0
615674,2176.0,4.0,3.0
615675,1508.0,3.0,3.0
615677,1583.0,3.0,3.0


In [92]:
#flags records with:
#- Less than 100 sq ft per bedroom: for example, 8 bedrooms in a 500 sq ft home.
#- Less than 50 sq ft per bathroom: for example, 6 bathrooms in a 250 sq ft home.

inconsistent = ( (filtered_data["BedroomsTotal"] * 100 > filtered_data["LivingArea"]) | (filtered_data["BathroomsTotalInteger"] * 50 > filtered_data["LivingArea"]) )
sum(inconsistent)

7

In [94]:
filtered_data.loc[inconsistent, ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger"]]

,LivingArea,BedroomsTotal,BathroomsTotalInteger
117204,100.0,4.0,3.0
252689,1289.0,3.0,153.0
269748,100.0,4.0,2.0
270243,100.0,4.0,2.0
309338,2.0,2.0,2.0
484268,105.0,3.0,2.0
527596,367.0,4.0,2.0


In [100]:
filtered_data2 = filtered_data.copy()
filtered_data2 = filtered_data2.loc[~inconsistent, ]
filtered_data2.shape

(303788, 85)

In [102]:
(303795-303788)/303795 #Log the count and percentage of records removed at each cleaning step 

2.3041853881729455e-05

### 1.2 Remove duplicate listings for the same property/transaction (same parcel ID or address + close date)

In [138]:
check = filtered_data2.copy()

# Standardize capitalization and spaces
check["address_clean"] = (
    check["UnparsedAddress"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# remove na
address_data = check.dropna(subset=["address_clean", "City", "CloseDate"])
address_data.shape

(303289, 86)

In [152]:
(303788-303289)/303788 #Log the count and percentage of records removed at each cleaning step 

0.0016425928608108287

In [148]:
duplicate_addresses = address_data[address_data.duplicated(subset=["address_clean", "City", "CloseDate"], keep="first")].sort_values(["address_clean", "City", "CloseDate"])
duplicate_addresses

,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,Date,BuyerAgentAOR,ListAgentAOR,OriginatingSystemName,OriginatingSystemSubName,address_clean
173996,NaN,False,NaN,NaN,False,1599000.0,1076991680,mark@socalpropertiesteams.com,2024-08-08,1565000.0,...,15147.0,NaN,NaN,NaN,202408,RanchoSoutheast,RanchoSoutheast,NaN,NaN,1025 verona drive
183898,NaN,False,NaN,NaN,False,1699000.0,1073239940,mark@socalpropertiesteams.com,2024-08-08,1565000.0,...,15147.0,NaN,NaN,NaN,202408,RanchoSoutheast,RanchoSoutheast,NaN,NaN,1025 verona drive
183899,NaN,False,NaN,NaN,False,1699000.0,1073239940,mark@socalpropertiesteams.com,2024-08-08,1565000.0,...,15147.0,NaN,NaN,NaN,202408,RanchoSoutheast,RanchoSoutheast,NaN,NaN,1025 verona drive
460358,"Carpet,Tile,Wood",False,NaN,NaN,False,950000.0,1119084897,walter@everhomere.com,2025-09-18,985000.0,...,6300.0,NaN,NaN,NaN,202509,SanDiego,SanDiego,NaN,NaN,10293 covina pl
446613,"Carpet,Tile",True,NaN,NaN,NaN,400000.0,1102577993,jason@iprore.com,2025-08-05,400000.0,...,7219.0,NaN,NaN,NaN,202508,Mlslistings,Mlslistings,NaN,NaN,10452 thorndale street
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179207,Wood,True,NaN,NaN,NaN,2470000.0,1076345514,ChrisJonesDRE@gmail.com,2024-08-05,2370000.0,...,24458.0,NaN,NaN,NaN,202408,BeverlyHillsGreaterLA,BeverlyHillsGreaterLA,NaN,NaN,944 appalachian
548494,Wood,True,NaN,NaN,NaN,10500000.0,1112705971,cristal@montecito-estate.com,2026-01-30,7650000.0,...,15682.0,NaN,NaN,NaN,202601,OutOfAreaBoard,OutOfAreaBoard,NaN,NaN,945 arbolado road
505115,"Carpet,Tile,Vinyl",True,NaN,NaN,False,1699700.0,1131058983,Ntaylor@kbhome.com,2025-11-26,1479700.0,...,8412.0,NaN,NaN,NaN,202511,NorthSanDiegoCounty,NorthSanDiegoCounty,NaN,NaN,951 visalia place
464393,Laminate,False,NaN,NaN,False,1599000.0,1116566597,Chilandrealtor@gmail.com,2025-09-22,1615000.0,...,7104.0,NaN,NaN,NaN,202509,OrangeCounty,OrangeCounty,NaN,NaN,9792 la arena circle


In [150]:
filtered_data3 = address_data.drop_duplicates(subset=["address_clean", "City", "CloseDate"], keep="first").copy()

print("Rows removed:", len(address_data) - len(filtered_data3))
filtered_data3.shape

Rows removed: 153


(303136, 86)

In [154]:
(303289-303136)/303289 #Log the count and percentage of records removed at each cleaning step 

0.0005044693345291125

I did not find the parcel ID, so I used address+close date to drop duplicates.

### 2. Select the variables that may be used later.

In [214]:
#Exclude: ListPrice / OriginalListPrice, Days on Market (DOM), Price reduction count/flag, Closing/settlement fields
#Use cautiously: Tax-assessed value, Listing agent / brokerage
#Keep: Square footage, beds/baths, lot size, year built, Latitude/longitude, neighborhood, school zone
#variables = ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude", 
#             "SubdivisionName", "ElementarySchoolDistrict", "MiddleOrJuniorSchoolDistrict", "HighSchoolDistrict", "PostalCode"]

numeric_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude"]
categorical_cols = ["SubdivisionName", "ElementarySchoolDistrict", "MiddleOrJuniorSchoolDistrict", "HighSchoolDistrict", "PostalCode"]

prep = filtered_data3.copy()
prep["record_id"] = np.arange(len(prep))
prep = prep[["record_id", "CloseDate", "ClosePrice"] + numeric_cols + categorical_cols].copy()
prep.shape

(303136, 15)

In [216]:
prep

,record_id,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,Latitude,Longitude,SubdivisionName,ElementarySchoolDistrict,MiddleOrJuniorSchoolDistrict,HighSchoolDistrict,PostalCode
28,0,2024-01-02,2100000.0,3736.0,4.0,4.0,11219.0,2019.0,35.277199,-120.646786,San Luis Obispo(380),NaN,NaN,San Luis Coastal Unified,93401
29,1,2024-01-22,1950000.0,2100.0,3.0,0.0,74487.6,1986.0,39.419080,-123.814842,NaN,NaN,NaN,NaN,95437
31,2,2024-01-31,2340000.0,2442.0,4.0,3.0,8712.0,1985.0,37.325454,-121.780303,NaN,NaN,NaN,Other,95148
37,3,2024-01-16,1060000.0,1917.0,3.0,3.0,13029.0,1965.0,34.107850,-118.219273,NaN,NaN,NaN,NaN,90065
38,4,2024-01-31,2150000.0,2200.0,3.0,2.0,6966.0,1961.0,37.589762,-122.399887,NaN,NaN,NaN,Other,94030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
615673,303131,2026-04-30,525000.0,944.0,3.0,1.0,7971.0,1947.0,34.265926,-118.425478,NaN,NaN,NaN,Los Angeles Unified,91331
615674,303132,2026-04-01,405000.0,2176.0,4.0,3.0,5200.0,1990.0,34.474548,-117.389973,NaN,NaN,NaN,Victor Valley Unified,92392
615675,303133,2026-04-01,215000.0,1508.0,3.0,3.0,10019.0,1990.0,38.985590,-122.694422,NaN,NaN,NaN,Konocti Unified,95424
615677,303134,2026-04-13,415000.0,1583.0,3.0,3.0,12096.0,2010.0,34.247895,-117.324666,Cedar Pines Park (CEPP),NaN,NaN,Rim of the World,92322


In [218]:
# Standardize categorical text and treat blank strings as missing.
for col in categorical_cols:
    prep[col] = (prep[col].astype("string")
                 .str.strip()
                 .str.lower()
                 .str.replace(r"\s+", " ", regex=True).replace("", pd.NA))

In [220]:
# Use the latest available month unless a complete month is specified.
prep["CloseMonth"] = prep["CloseDate"].dt.to_period("M")
test_month = prep["CloseMonth"].max()
print("Test month:", test_month)

Test month: 2026-04


In [222]:
# Compare the available sample sizes for different training windows.
window_options = [3, 6, 12, 18, 24]
window_rows = []
for months in window_options:
    candidate = prep.loc[(prep["CloseMonth"] >= test_month - months) & (prep["CloseMonth"] < test_month)]
    window_rows.append([months, str(test_month - months), str(test_month - 1), candidate["CloseMonth"].nunique(), len(candidate)])

window_summary = pd.DataFrame(window_rows, columns=["X_months", "train_start", "train_end", "observed_months", "train_rows"])
window_summary["complete_window"] = window_summary["observed_months"].eq(window_summary["X_months"])
window_summary

,X_months,train_start,train_end,observed_months,train_rows,complete_window
0,3,2026-01,2026-03,3,27403,True
1,6,2025-10,2026-03,6,59024,True
2,12,2025-04,2026-03,12,127907,True
3,18,2024-10,2026-03,18,187820,True
4,24,2024-04,2026-03,24,262097,True


In [224]:
X = 12 # X can be changed
train = prep.loc[(prep["CloseMonth"] >= test_month - X) & (prep["CloseMonth"] < test_month)].copy()
test = prep.loc[prep["CloseMonth"] == test_month].copy()

print("Training rows:", len(train), ", Test rows:", len(test))
print("Training period:", train["CloseDate"].min(), "to", train["CloseDate"].max())

Training rows: 127907 , Test rows: 12190
Training period: 2025-04-01 00:00:00 to 2026-03-31 00:00:00


In [226]:
# Learn price cutoffs from the training window only.
train_price_low = train["ClosePrice"].quantile(0.005)
train_price_high = train["ClosePrice"].quantile(0.995)
train_rows_before = len(train)
train = train.loc[train["ClosePrice"].between(train_price_low, train_price_high)].copy()

print(f"Training price bounds: ({train_price_low}, {train_price_high})")
print("Training rows removed:", train_rows_before - len(train))

Training price bounds: (185000.0, 8700000.0)
Training rows removed: 1263


In [228]:
# Apply the same frozen cutoffs to the test month without discarding the full test set.
test_price_mask = test["ClosePrice"].between(train_price_low, train_price_high)

print("Test rows within training bounds:", test_price_mask.sum(), "/", len(test))

Test rows within training bounds: 12069 / 12190


In [230]:
# Record the count and percentage affected by the price filter.
price_filter_summary = pd.DataFrame({"split": ["train", "test"], 
                                     "before": [train_rows_before, len(test)],
                                     "within_bounds": [len(train), int(test_price_mask.sum())]})

price_filter_summary["outside_bounds"] = price_filter_summary["before"] - price_filter_summary["within_bounds"]
price_filter_summary["outside_pct"] = 100 * price_filter_summary["outside_bounds"] / price_filter_summary["before"]
price_filter_summary

,split,before,within_bounds,outside_bounds,outside_pct
0,train,127907,126644,1263,0.987436
1,test,12190,12069,121,0.992617


### 3. Handle missing values

In [233]:
# Examine missingness after splitting and filtering the training data.
missing_summary = pd.DataFrame({
    "train_missing": train[numeric_cols + categorical_cols].isna().sum(),
    "train_missing_pct": train[numeric_cols + categorical_cols].isna().mean() * 100,
    "test_missing": test[numeric_cols + categorical_cols].isna().sum()})
missing_summary

,train_missing,train_missing_pct,test_missing
LivingArea,0,0.000000,0
BedroomsTotal,0,0.000000,0
BathroomsTotalInteger,16,0.012634,0
LotSizeSquareFeet,0,0.000000,0
YearBuilt,57,0.045008,7
Latitude,115,0.090806,60
Longitude,115,0.090806,60
SubdivisionName,82852,65.421181,7981
ElementarySchoolDistrict,126644,100.000000,12190
MiddleOrJuniorSchoolDistrict,126644,100.000000,12190


In [235]:
# Drop columns using training data only; keep ZIP as a core location feature.
empty_numeric = [col for col in numeric_cols if train[col].isna().all()]
high_missing_cat = [col for col in categorical_cols if col != "PostalCode" and train[col].isna().mean() > 0.80]

num_used = [col for col in numeric_cols if col not in empty_numeric]
cat_used = [col for col in categorical_cols if col not in high_missing_cat]

print("Dropped numeric columns:", empty_numeric)
print("Dropped categorical columns:", high_missing_cat)

Dropped numeric columns: []
Dropped categorical columns: ['ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict']


In [248]:
# Keep separate copies for filling missing values and exporting readable CSV files.
train_clean = train[["record_id", "CloseDate", "ClosePrice"] + num_used + cat_used].copy()
test_clean = test_all[["record_id", "CloseDate", "ClosePrice"] + num_used + cat_used].copy()

# Add missing indicators before imputation.
flag_cols = []
for col in num_used:
    flag = col + "_missing"
    train_clean[flag] = train_clean[col].isna().astype(int)
    test_clean[flag] = test_clean[col].isna().astype(int)
    flag_cols.append(flag)

In [250]:
# Use training medians for both training and test data.
train_medians = train_clean[num_used].median()
train_clean[num_used] = train_clean[num_used].fillna(train_medians)
test_clean[num_used] = test_clean[num_used].fillna(train_medians)
train_medians

LivingArea               1814.000000
BedroomsTotal               3.000000
BathroomsTotalInteger       2.000000
LotSizeSquareFeet        7269.000000
YearBuilt                1976.000000
Latitude                   34.089764
Longitude                -118.042011
dtype: float64

In [252]:
# Treat missing categories as an explicit category.
train_clean[cat_used] = train_clean[cat_used].fillna("__missing__")
test_clean[cat_used] = test_clean[cat_used].fillna("__missing__")

# Confirm that all selected features are complete.
train_clean[num_used + cat_used + flag_cols].isna().sum()

LivingArea                       0
BedroomsTotal                    0
BathroomsTotalInteger            0
LotSizeSquareFeet                0
YearBuilt                        0
Latitude                         0
Longitude                        0
SubdivisionName                  0
HighSchoolDistrict               0
PostalCode                       0
LivingArea_missing               0
BedroomsTotal_missing            0
BathroomsTotalInteger_missing    0
LotSizeSquareFeet_missing        0
YearBuilt_missing                0
Latitude_missing                 0
Longitude_missing                0
dtype: int64

### 4. Encode categorical features

In [254]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy import sparse
import joblib

# Learn categories from training data only.
encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, max_categories=100, sparse_output=True)
train_cat = encoder.fit_transform(train_clean[cat_used])
test_cat = encoder.transform(test_clean[cat_used])
cat_names = encoder.get_feature_names_out(cat_used).tolist()
print("Encoded categorical columns:", len(cat_names))

Encoded categorical columns: 300


In [256]:
### 5. Standardize numerical features

In [258]:
# Fit the scaler on training numerical features only.
scaler = StandardScaler()
train_num = scaler.fit_transform(train_clean[num_used])
test_num = scaler.transform(test_clean[num_used])

In [260]:
# Combine scaled numeric features, missing flags, and encoded categories.
X_train = sparse.hstack([train_num, train_clean[flag_cols].to_numpy(), train_cat], format="csr")
X_test = sparse.hstack([test_num, test_clean[flag_cols].to_numpy(), test_cat], format="csr")
feature_names = num_used + flag_cols + cat_names

# Keep targets separate; also prepare the AVM price-range test subset.
y_train = train_clean["ClosePrice"].copy()
y_test = test_clean["ClosePrice"].copy()
X_test_avm = X_test[test_price_mask.to_numpy()]
y_test_avm = y_test.loc[test_price_mask]

In [266]:
# Check the time split, missing values, and consistent feature dimensions.
assert train_clean["CloseDate"].max() < test_clean["CloseDate"].min()
assert set(train_clean["record_id"]).isdisjoint(test_clean["record_id"])
assert X_train.shape[1] == X_test.shape[1] == len(feature_names)
assert np.isfinite(X_train.data).all() and np.isfinite(X_test.data).all()
assert not test_clean[num_used + cat_used + flag_cols].isna().any().any()
assert "ClosePrice" not in feature_names and "CloseDate" not in feature_names
print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("X_test_avm:", X_test_avm.shape)

X_train: (126644, 314)   X_test: (12190, 314)
X_test_avm: (12069, 314)


### 6. Export CSV

In [271]:
# Create the output folder and record each row's split.
output_folder = Path("week3_output")
output_folder.mkdir(parents=True, exist_ok=True)
train_export = train_clean.assign(split="train", in_train_price_range=True)
test_export = test_clean.assign(split="test", in_train_price_range=test_price_mask)
cleaned_data = pd.concat([train_export, test_export], ignore_index=True)

In [273]:
# Save readable data in original numerical units.
cleaned_data.to_csv(output_folder / "cleaned_data.csv", index=False)
train_export.to_csv(output_folder / "train_cleaned.csv", index=False)
test_export.to_csv(output_folder / "test_cleaned.csv", index=False)
test_export.loc[test_price_mask].to_csv(output_folder / "test_avm_range.csv", index=False)

In [277]:
# Save sparse numeric inputs without expanding them into large dense tables.
sparse.save_npz(output_folder / "X_train.npz", X_train)
sparse.save_npz(output_folder / "X_test_all.npz", X_test)
sparse.save_npz(output_folder / "X_test_avm.npz", X_test_avm)
pd.Series(feature_names, name="feature").to_csv(output_folder / "feature_names.csv", index=False)

In [279]:
# Save preprocessing settings for reuse; no prediction model is fitted here.
preprocessing_objects = {"X_months": X, "test_month": str(test_month),
                         "num_used": num_used, "cat_used": cat_used, "flag_cols": flag_cols,
                         "train_medians": train_medians, "encoder": encoder, "scaler": scaler,
                         "feature_names": feature_names,
                         "price_bounds": [train_price_low, train_price_high]}
joblib.dump(preprocessing_objects, output_folder / "preprocessing_objects.joblib")

['week3_output/preprocessing_objects.joblib']

In [281]:
# Save preprocessing summaries for documentation.
window_summary.to_csv(output_folder / "training_window_options.csv", index=False)
missing_summary.to_csv(output_folder / "missing_summary.csv", index_label="feature")
price_filter_summary.to_csv(output_folder / "price_filter_summary.csv", index=False)
print("Saved files to:", output_folder.resolve())
cleaned_data.head()

Saved files to: /Users/jingying/Documents/IDX Exchange/py/week3_output


,record_id,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,Latitude,Longitude,...,PostalCode,LivingArea_missing,BedroomsTotal_missing,BathroomsTotalInteger_missing,LotSizeSquareFeet_missing,YearBuilt_missing,Latitude_missing,Longitude_missing,split,in_train_price_range
0,163039,2025-04-30,2700000.0,4154.0,4.0,5.0,20909.0,2018.0,33.651150,-116.244673,...,92253,0,0,0,0,0,0,0,train,True
1,163040,2025-04-30,3250000.0,3104.0,3.0,4.0,20251.0,2004.0,33.057843,-117.182771,...,92091,0,0,0,0,0,0,0,train,True
2,163041,2025-04-30,1275000.0,1265.0,2.0,2.0,6752.0,1951.0,34.155882,-118.451233,...,91403,0,0,0,0,0,0,0,train,True
3,163042,2025-04-14,1701000.0,3070.0,4.0,3.0,8747.0,1984.0,32.850572,-117.187512,...,92122,0,0,0,0,0,0,0,train,True
4,163043,2025-04-29,1750000.0,3025.0,3.0,3.0,18000.0,1977.0,33.418740,-117.588871,...,92672,0,0,0,0,0,0,0,train,True
